In [1]:
!pip install sentence-transformers
!pip install protobuf
!pip install deep_translator
!pip install pandas

     ---------------------------------------- 0.0/44.1 kB ? eta -:--:--
     --------- ------------------------------ 10.2/44.1 kB ? eta -:--:--
     -------------------------------------- 44.1/44.1 kB 538.0 kB/s eta 0:00:00
     ---------------------------------------- 0.0/57.6 kB ? eta -:--:--
     ---------------------------------------- 57.6/57.6 kB 1.5 MB/s eta 0:00:00
  Using cached scikit_learn-1.5.2-cp311-cp311-win_amd64.whl.metadata (13 kB)
  Using cached scipy-1.14.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached pillow-11.0.0-cp311-cp311-win_amd64.whl.metadata (9.3 kB)
  Using cached PyYAML-6.0.2-cp311-cp311-win_amd64.whl.metadata (2.1 kB)
  Using cached requests-2.32.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached jinja2-3.1.4-py3-none-any.whl.metadata (2.6 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached numpy-2.1.3-cp311-cp311-win_amd64.whl.metadata (60 kB)
     ---------------------------------------- 0.0/41.5 kB ? eta -:--


[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached protobuf-5.28.3-cp310-abi3-win_amd64.whl.metadata (592 bytes)
Using cached protobuf-5.28.3-cp310-abi3-win_amd64.whl (431 kB)



[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached beautifulsoup4-4.12.3-py3-none-any.whl.metadata (3.8 kB)
   ---------------------------------------- 0.0/42.3 kB ? eta -:--:--
   ---------------------------------------- 42.3/42.3 kB 1.0 MB/s eta 0:00:00
Using cached beautifulsoup4-4.12.3-py3-none-any.whl (147 kB)



[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.6 MB 9.4 MB/s eta 0:00:02
   ------ --------------------------------- 1.9/11.6 MB 20.6 MB/s eta 0:00:01
   ----------- ---------------------------- 3.3/11.6 MB 26.6 MB/s eta 0:00:01
   ------------ --------------------------- 3.7/11.6 MB 19.6 MB/s eta 0:00:01
   ------------- -------------------------- 3.9/11.6 MB 17.8 MB/s eta 0:00:01
   -------------- ------------------------- 4.1/11.6 MB 15.5 MB/s eta 0:00:01
   --------------- ------------------------ 4.4/11.6 MB 13.4 MB/s eta 0:00:01
   --------------- ------------------------ 4.6/11.6 MB 12.8 MB/s eta 0:00:01
   ---------------- ----------------------- 4.8/11.6 MB 12.2 MB/s eta 0:00:01
   ----------------- ---------------------- 4.9/11.6 MB 10.9 MB/s eta 0:00:01
   ----------------- ---------------------- 5.0/11.6 MB 10.1 MB/s eta 0:00:01
   ----------------- ---------------------- 5.1/11.6 MB 9.4 MB/s eta 0:00


[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import time

import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from torch import nn, optim, tensor
from torch.nn import functional as F

In [4]:
class Classifier(nn.Module):
    def __init__(self, embedding_dim, num_labels, dropout):
        super(Classifier, self).__init__()
        self.embedding_dim = embedding_dim
        self.num_labels = num_labels
        self.dropout = dropout

        self.dp = nn.Dropout(self.dropout)
        self.ff = nn.Linear(self.embedding_dim, self.num_labels)

    def forward(self, input_embeddings):
        tensor = self.dp(input_embeddings)
        tensor = self.ff(tensor)
        return tensor, F.softmax(tensor, dim=-1)


In [5]:
class Batcher(object):
    def __init__(self, data_x, data_y, batch_size):
        self.data_x = data_x
        self.data_y = data_y
        self.batch_size = batch_size
        self.n_samples = data_x.shape[0]
        self.indices = torch.randperm(self.n_samples)
        self.ptr = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.ptr > self.n_samples:
            self.ptr = 0
            self.indices = torch.randperm(self.n_samples)
            raise StopIteration
        else:
            batch_indices = self.indices[self.ptr:self.ptr+self.batch_size]
            self.ptr += self.batch_size
            return self.data_x[batch_indices], self.data_y[batch_indices]


In [8]:
data_df = pd.read_csv("C:\Programming\Text Mining\data-en-hi-de-fr.csv\data-en-hi-de-fr.csv")

data_df.dropna(inplace=True)
data_df.drop_duplicates(inplace=True)
data_df.rename(columns={
    "Category": "labels",
    "Message": "text"
}, inplace=True)

le = LabelEncoder()
le.fit(data_df.labels)
data_df["labels"] = le.transform(data_df.labels)


In [9]:
train_x, test_x, train_y, test_y = \
    train_test_split(data_df.text, data_df.labels, stratify=data_df.labels, test_size=0.15,
                     random_state=123)

train_x_de, test_x_de, train_y_de, test_y_de = \
    train_test_split(data_df.text_hi, data_df.labels, stratify=data_df.labels, test_size=0.15,
                     random_state=123)

sentences = train_x.tolist()
test_sentences = test_x_de.tolist()

labels = torch.tensor(train_y.tolist())
test_labels = torch.tensor(test_y_de.tolist())


In [10]:
# encoder = SentenceTransformer('distilbert-base-nli-mean-tokens')
encoder = SentenceTransformer('quora-distilbert-multilingual')
print('Encoding segments...')
start = time.time()
embedding = encoder.encode(sentences, convert_to_tensor=True)
test_sentences_embedding = encoder.encode(test_sentences, convert_to_tensor=True)
print(f"Encoding completed in {time.time() - start} seconds.")


c:\Programming\Text Mining\tmenv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\models--sentence-transformers--quora-distilbert-multilingual. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Encoding segments...
Encoding completed in 97.66729497909546 seconds.


In [11]:
train_batcher = Batcher(embedding, labels, batch_size=16)

num_samples, embeddings_dim = embedding.size()
n_labels = labels.unique().shape[0]

classifier = Classifier(embeddings_dim, n_labels, dropout=0.01)


In [12]:
optimizer = optim.Adam(classifier.parameters())
loss_fn = nn.CrossEntropyLoss()


In [13]:
for e in range(10):
    total_loss = 0
    for batch in train_batcher:
        x, y = batch
        optimizer.zero_grad()
        model_output, prob = classifier(x)
        loss = loss_fn(model_output, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'epoch:{e}, total_loss:{total_loss}')


epoch:0, total_loss:63.537392100319266
epoch:1, total_loss:34.82884030044079
epoch:2, total_loss:27.670757913030684
epoch:3, total_loss:23.588982296176255
epoch:4, total_loss:20.794334064237773
epoch:5, total_loss:19.00357951084152
epoch:6, total_loss:17.887130048591644
epoch:7, total_loss:16.70992535050027
epoch:8, total_loss:15.752393983770162
epoch:9, total_loss:14.920220942934975


In [14]:
with torch.no_grad():
    model_output, prob = classifier(test_sentences_embedding)
    predictions = torch.argmax(prob, dim=-1)
    results = classification_report(predictions, test_labels)
    print(results)


              precision    recall  f1-score   support

           0       1.00      0.93      0.96       728
           1       0.45      0.93      0.61        46

    accuracy                           0.93       774
   macro avg       0.72      0.93      0.78       774
weighted avg       0.96      0.93      0.94       774

